# Advanced: Multi-Strategy Fake Organization Generation with PAMOLA.CORE

**Goal**: Master all 3 organization generation strategies using operation.execute()

**What you'll learn:**
- **Strategy 1**: Region-based generation (English, Vietnamese, Russian naming)
- **Strategy 2**: Industry-specific organization generation
- **Strategy 3**: Type-aware with custom dictionaries
- Advanced consistency mechanisms (PRGN vs mapping)
- Multi-field organization processing
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of PRGN consistency
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── fake_data/
        └── 02_fake_organization_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Import required modules
try:
    from pamola_core.fake_data.operations.organization_op import FakeOrganizationOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Restart your Python kernel/notebook")
    print("   2. Verify: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 8 columns)
- Sample records (first 5 rows)
- Column statistics (types, unique counts)
- Organization types and regions distribution

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'organization_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Organization types for variety
    org_types = ['corporation', 'llc', 'tech', 'manufacturing', 'retail', 
                 'financial', 'healthcare', 'nonprofit', 'government', 'education']
    
    # Regions for multi-regional testing
    regions = ['en', 'eu', 'asia']
    region_weights = [0.5, 0.3, 0.2]  # English dominant
    
    # Industries
    industries = [
        'Technology', 'Manufacturing', 'Retail', 'Finance', 'Healthcare',
        'Education', 'Construction', 'Transportation', 'Energy', 'Media',
        'Hospitality', 'Real Estate', 'Legal', 'Consulting', 'Telecommunications'
    ]
    
    # Generate organization names (will be replaced)
    base_names = [
        'Global Solutions', 'Tech Corp', 'Industries Inc', 'Systems Ltd',
        'Enterprises', 'Group Holdings', 'International', 'Partners LLC',
        'Innovations', 'Services Group', 'Manufacturing Co', 'Technologies',
        'Financial Group', 'Healthcare Systems', 'Construction Ltd'
    ]
    
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'company_primary': [f"{np.random.choice(['Acme', 'Global', 'Metro', 'United', 'First'])} {np.random.choice(base_names)}" 
                           for _ in range(n_records)],
        'company_subsidiary': [f"{np.random.choice(['East', 'West', 'North', 'South', 'Central'])} {np.random.choice(['Division', 'Branch', 'Unit', 'Office'])}" 
                              for _ in range(n_records)],
        'company_parent': [f"{np.random.choice(['Alpha', 'Beta', 'Gamma', 'Delta', 'Omega'])} Holdings" 
                          for _ in range(n_records)],
        'organization_type': np.random.choice(org_types, n_records),
        'region': np.random.choice(regions, n_records, p=region_weights),
        'industry': np.random.choice(industries, n_records),
        'employee_count': np.random.randint(10, 10000, n_records)
    })
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:25s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:25s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

print(f"\n🏢 Organization Distribution:")
print(f"  Types: {df['organization_type'].value_counts().to_dict()}")
print(f"  Regions: {df['region'].value_counts().to_dict()}")
print(f"  Top Industries: {dict(df['industry'].value_counts().head(5))}")

print(f"\n💡 Perfect dataset for testing all 3 strategies!")

## Step 3: Prepare Custom Organization Dictionaries

**How to use:**
- Run to check/create custom organization dictionaries
- Uses existing files if found
- Creates example dictionaries if missing

**What you'll see:**
- File status for each dictionary (✓ found or 🔧 created)
- Dictionary structure summaries
- Organization type counts per region
- Prefix/suffix examples

**Dictionaries created:**
- **organization_names.json**: Base organization names by type and region
- **organization_prefixes.json**: Regional prefixes (Global, Euro, Asia)
- **organization_suffixes.json**: Regional suffixes (Inc, GmbH, Ltd)

**Note:** Edit files to match your data structure before processing

In [ ]:
# Define dictionary paths (in examples directory)
examples_dir = project_root / 'examples'
dict_dir = examples_dir / 'data_examples'

# Dictionary file paths
names_path = dict_dir / 'organization_names.json'
prefixes_path = dict_dir / 'organization_prefixes.json'
suffixes_path = dict_dir / 'organization_suffixes.json'

# Check if dictionaries exist
all_exist = all([names_path.exists(), prefixes_path.exists(), suffixes_path.exists()])

if all_exist:
    print("✓ Found all custom dictionaries")
    print(f"  - {names_path.name}")
    print(f"  - {prefixes_path.name}")
    print(f"  - {suffixes_path.name}")
    print("\n💡 Using existing dictionaries for Strategy 3")
else:
    print("🔧 Creating custom organization dictionaries...\n")
    
    # 1. Organization Names Dictionary
    org_names = {
        "tech": {
            "en": ["Technologies", "Systems", "Solutions", "Software", "Digital", "Cloud", "Data"],
            "eu": ["Technologie", "Systeme", "Digital", "Software"],
            "asia": ["Technology", "Digital", "Systems", "Innovation"]
        },
        "manufacturing": {
            "en": ["Manufacturing", "Industries", "Production", "Engineering"],
            "eu": ["Industrie", "Produktion", "Fertigung"],
            "asia": ["Industries", "Manufacturing", "Production"]
        },
        "financial": {
            "en": ["Capital", "Finance", "Investments", "Banking", "Wealth"],
            "eu": ["Kapital", "Finanz", "Bank"],
            "asia": ["Capital", "Finance", "Investment"]
        },
        "retail": {
            "en": ["Retail", "Commerce", "Stores", "Markets"],
            "eu": ["Handel", "Markt"],
            "asia": ["Retail", "Commerce", "Trading"]
        },
        "healthcare": {
            "en": ["Healthcare", "Medical", "Health", "Care"],
            "eu": ["Gesundheit", "Medizin"],
            "asia": ["Medical", "Healthcare", "Health"]
        }
    }
    
    # 2. Prefixes Dictionary
    org_prefixes = {
        "tech": {
            "en": ["Global", "Advanced", "Smart", "Next"],
            "eu": ["Euro", "Continental", "United"],
            "asia": ["Asia", "Pacific", "Orient"]
        },
        "financial": {
            "en": ["First", "Premier", "Elite"],
            "eu": ["Union", "Central", "Federal"],
            "asia": ["Eastern", "Pacific", "Asia"]
        },
        "general": {
            "en": ["Global", "International", "United"],
            "eu": ["Euro", "Continental"],
            "asia": ["Asia", "Pacific"]
        }
    }
    
    # 3. Suffixes Dictionary
    org_suffixes = {
        "corporation": {
            "en": ["Inc", "Corp", "Corporation"],
            "eu": ["GmbH", "AG", "SA"],
            "asia": ["Ltd", "Co., Ltd", "Corporation"]
        },
        "llc": {
            "en": ["LLC", "Limited"],
            "eu": ["SRL", "Limited"],
            "asia": ["Limited", "Pte Ltd"]
        },
        "general": {
            "en": ["Group", "Holdings", "Enterprises"],
            "eu": ["Group", "Holding"],
            "asia": ["Group", "Holdings"]
        }
    }
    
    # Save dictionaries
    try:
        dict_dir.mkdir(parents=True, exist_ok=True)
        
        with open(names_path, 'w') as f:
            json.dump(org_names, f, indent=2)
        print(f"✓ Created: {names_path.name}")
        
        with open(prefixes_path, 'w') as f:
            json.dump(org_prefixes, f, indent=2)
        print(f"✓ Created: {prefixes_path.name}")
        
        with open(suffixes_path, 'w') as f:
            json.dump(org_suffixes, f, indent=2)
        print(f"✓ Created: {suffixes_path.name}")
        
    except PermissionError:
        print(f"⚠️  Cannot save dictionaries (files may be open)")

# Display dictionary statistics
print("\n📊 Dictionary Statistics:")
print("=" * 80)

with open(names_path, 'r') as f:
    names_dict = json.load(f)
    total_names = sum(len(regions) for types in names_dict.values() for regions in types.values())
    print(f"  Organization Names: {len(names_dict)} types, {total_names} total names")
    print(f"    Types: {', '.join(names_dict.keys())}")

with open(prefixes_path, 'r') as f:
    prefixes_dict = json.load(f)
    total_prefixes = sum(len(regions) for types in prefixes_dict.values() for regions in types.values())
    print(f"\n  Prefixes: {len(prefixes_dict)} categories, {total_prefixes} total prefixes")

with open(suffixes_path, 'r') as f:
    suffixes_dict = json.load(f)
    total_suffixes = sum(len(regions) for types in suffixes_dict.values() for regions in types.values())
    print(f"\n  Suffixes: {len(suffixes_dict)} categories, {total_suffixes} total suffixes")

print("\n💡 Edit these files to customize organization generation!")

## Step 4: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `"strategy1_field": "company_primary"` (region-based)
   - `"strategy2_field": "company_subsidiary"` (industry-specific)
   - `"strategy3_field": "company_parent"` (type-aware with dictionaries)
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Unique organization counts per field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "company_primary",     # Region-based generation
    "strategy2_field": "company_subsidiary",  # Industry-specific generation
    "strategy3_field": "company_parent"       # Type-aware with dictionaries
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    print(f"  ✓ {strategy:20s}: '{field_name}' ({df[field_name].nunique()} unique values)")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_org_001",
    task_type="multi_strategy_organization",
    description="Multi-strategy fake organization generation",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Region-Based Generation

**How to use:**
- Uses region field to generate appropriate organization names
- Automatically adapts naming conventions (English, Vietnamese, Russian)
- Run to execute region-based strategy

**Key parameters:**
- `organization_type='general'`: General business organizations
- `region='en'`: Default region (overridden by region_field)
- `region_field='region'`: Uses dataset's region column
- `preserve_type=True`: Maintains organization type characteristics
- `mode='ENRICH'`: Adds new column, keeps original

**What you'll see:**
- Configuration summary
- Progress bar: validation → generation → metrics → viz → save
- Completion time (3-5 seconds for 1000 records)
- Multi-regional organization names

**Note:** Creates `company_primary_regional` with region-appropriate names

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: REGION-BASED GENERATION")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Region-based",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = FakeOrganizationOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy1_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy1_field']}_regional",
    output_format='csv',
    
    # Organization generation
    organization_type='general',
    region='en',  # Default, overridden by region_field
    region_field='region',  # Use dataset's region column
    preserve_type=True,
    type_field='organization_type',
    
    # Prefix/Suffix probabilities
    add_prefix_probability=0.4,
    add_suffix_probability=0.6,
    
    # Consistency
    consistency_mechanism='prgn',
    key='region-strategy-2025',
    context_salt='regional-orgs',
    
    # Metrics
    collect_type_distribution=True,
    detailed_metrics=True,
    
    # Processing
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"  Region handling: Uses '{operation_s1.region_field}' field")
print(f"  Type preservation: {operation_s1.preserve_type}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_regional',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_regional' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    result_df_s1 = pd.read_csv(output_files_s1[0])
    field_s1 = FIELD_CONFIG["strategy1_field"]
    output_field_s1 = f"{field_s1}_regional"
    
    print(f"\n📊 Sample Regional Organizations:")
    sample_by_region = result_df_s1.groupby('region').head(2)
    for _, row in sample_by_region.iterrows():
        print(f"  [{row['region']:4s}] {row[field_s1]:<40} → {row[output_field_s1]}")

## STRATEGY 2: Industry-Specific Generation

**How to use:**
- Generates organizations appropriate for specific industries
- Uses industry field for contextual generation
- Run to execute industry-specific strategy

**Key parameters:**
- `organization_type='general'`: Base type
- `industry=None`: Uses dataset's industry column
- `preserve_type=True`: Maintains characteristics
- `consistency_mechanism='mapping'`: Deterministic mapping
- `save_mapping=True`: Saves organization mappings

**What you'll see:**
- Configuration confirmation
- Progress bar: validation → generation → metrics → viz → save
- Completion time (3-5 seconds)
- Industry-appropriate organization names
- Mapping file saved for reproducibility

**Note:** Creates `company_subsidiary_industry` with industry-contextual names

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: INDUSTRY-SPECIFIC GENERATION")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Industry-specific",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = FakeOrganizationOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy2_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy2_field']}_industry",
    output_format='csv',
    
    # Organization generation
    organization_type='general',
    region='en',
    preserve_type=True,
    type_field='organization_type',
    industry=None,  # Will use context from data
    
    # Prefix/Suffix
    add_prefix_probability=0.3,
    add_suffix_probability=0.5,
    
    # Consistency with mapping
    consistency_mechanism='mapping',
    save_mapping=True,
    
    # Metrics
    collect_type_distribution=True,
    detailed_metrics=True,
    
    # Processing
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"  Industry-aware: Using data context")
print(f"  Consistency: {operation_s2.consistency_mechanism}")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_industry',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results (NEWEST file)
output_files_s2 = sorted(
    list((task_dir / 'strategy2_industry' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    result_df_s2 = pd.read_csv(output_files_s2[0])
    field_s2 = FIELD_CONFIG["strategy2_field"]
    output_field_s2 = f"{field_s2}_industry"
    
    print(f"\n📊 Sample Industry-Specific Organizations:")
    sample_by_industry = result_df_s2.groupby('industry').head(1)
    for _, row in sample_by_industry.iterrows():
        print(f"  [{row['industry']:20s}] {row[output_field_s2]}")

## STRATEGY 3: Type-Aware with Custom Dictionaries

**How to use:**
- Uses custom dictionaries from Step 3
- Type-aware generation with regional adaptation
- Run to execute dictionary-based strategy

**Key parameters:**
- `dictionaries`: Custom organization names by type/region
- `prefixes`: Custom prefixes by type/region
- `suffixes`: Custom suffixes by type/region
- `preserve_type=True`: Respects organization types
- `region_field='region'`: Uses dataset regions

**What you'll see:**
- Configuration confirmation with dictionary info
- Progress bar: validation → dictionary load → generation → metrics → viz → save
- Completion time (3-5 seconds)
- Type-appropriate, region-aware organization names

**Note:** Creates `company_parent_custom` with dictionary-based generation

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: TYPE-AWARE WITH CUSTOM DICTIONARIES")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Custom dictionaries",
    unit="steps",
    track_memory=True,
    level=0
)

# Load custom dictionaries
with open(names_path, 'r') as f:
    custom_names = json.load(f)
with open(prefixes_path, 'r') as f:
    custom_prefixes = json.load(f)
with open(suffixes_path, 'r') as f:
    custom_suffixes = json.load(f)

operation_s3 = FakeOrganizationOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy3_field"],
    mode='ENRICH',
    
    # Output configuration
    output_field_name=f"{FIELD_CONFIG['strategy3_field']}_custom",
    output_format='csv',
    
    # Custom dictionaries
    dictionaries=custom_names,
    prefixes=custom_prefixes,
    suffixes=custom_suffixes,
    
    # Organization generation
    organization_type='general',
    region='en',
    region_field='region',
    preserve_type=True,
    type_field='organization_type',
    
    # Prefix/Suffix probabilities
    add_prefix_probability=0.5,
    add_suffix_probability=0.7,
    
    # Consistency
    consistency_mechanism='prgn',
    key='custom-dict-2025',
    context_salt='custom-orgs',
    
    # Metrics
    collect_type_distribution=True,
    detailed_metrics=True,
    
    # Processing
    use_vectorization=True,
    parallel_processes=2,
    use_cache=False,
    
    # Execution
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"  Custom dictionaries: {len(custom_names)} types")
print(f"  Custom prefixes: {len(custom_prefixes)} categories")
print(f"  Custom suffixes: {len(custom_suffixes)} categories")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_custom',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results (NEWEST file)
output_files_s3 = sorted(
    list((task_dir / 'strategy3_custom' / 'output').glob('*.csv')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    result_df_s3 = pd.read_csv(output_files_s3[0])
    field_s3 = FIELD_CONFIG["strategy3_field"]
    output_field_s3 = f"{field_s3}_custom"
    
    print(f"\n📊 Sample Custom Dictionary Organizations:")
    sample_custom = result_df_s3.head(10)
    for _, row in sample_custom.iterrows():
        print(f"  [{row['organization_type']:12s}] {row[output_field_s3]}")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and characteristics

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Organization generation rate (orgs/second)
- Strategy characteristics comparison

**Note:** All strategies process 1000 organizations, differences in time reflect complexity

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Regional):      {elapsed_s1:6.2f}s ({1000/elapsed_s1:6.0f} orgs/sec)")
print(f"  Strategy 2 (Industry):      {elapsed_s2:6.2f}s ({1000/elapsed_s2:6.0f} orgs/sec)")
print(f"  Strategy 3 (Custom Dict):   {elapsed_s3:6.2f}s ({1000/elapsed_s3:6.0f} orgs/sec)")
print(f"  Total:                      {elapsed_s1 + elapsed_s2 + elapsed_s3:6.2f}s")

print(f"\n📊 Strategy Characteristics:")
print(f"\n  Strategy 1 - Region-Based:")
print(f"    • Multi-regional support (English, Vietnamese, Russian)")
print(f"    • Automatic region detection from data")
print(f"    • PRGN consistency for reproducibility")
print(f"    • Best for: Global datasets with regional diversity")

print(f"\n  Strategy 2 - Industry-Specific:")
print(f"    • Industry-contextual generation")
print(f"    • Mapping-based consistency (deterministic)")
print(f"    • Saves mappings for audit trail")
print(f"    • Best for: Industry-focused datasets, compliance needs")

print(f"\n  Strategy 3 - Custom Dictionaries:")
print(f"    • Full control over organization names")
print(f"    • Type-aware with regional adaptation")
print(f"    • Custom prefix/suffix patterns")
print(f"    • Best for: Specialized domains, specific naming conventions")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Auto-loads NEWEST files from each category
- Excludes system files automatically

**What you'll see (per strategy):**
1. **Metrics JSON**: Generation performance, type distribution, quality metrics
2. **Organization Mapping**: Original → Synthetic mappings (Strategy 2 only)
3. **Visualizations**: Distribution charts displayed inline (latest batch)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_regional', 'Strategy 1: Region-Based'),
    ('strategy2_industry', 'Strategy 2: Industry-Specific'),
    ('strategy3_custom', 'Strategy 3: Custom Dictionaries')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                # Performance
                if 'performance' in metrics:
                    perf = metrics['performance']
                    print(f"\n   Performance:")
                    print(f"     Generation time: {perf.get('generation_time', 0):.4f}s")
                    print(f"     Records/second:  {perf.get('records_per_second', 0):,}")
                    print(f"     Memory usage:    {perf.get('memory_usage_mb', 0):.2f} MB")
                
                # Organization generator info
                if 'organization_generator' in metrics:
                    og = metrics['organization_generator']
                    print(f"\n   Organization Generator:")
                    print(f"     Type:            {og.get('organization_type', 'N/A')}")
                    print(f"     Region:          {og.get('region', 'N/A')}")
                    print(f"     Preserve type:   {og.get('preserve_type', False)}")
                    
                    # Type distribution
                    if 'type_distribution' in og:
                        td = og['type_distribution']
                        print(f"\n     Type Distribution:")
                        print(f"       Total orgs:    {td.get('total_organizations', 0):,}")
                        print(f"       Unique types:  {td.get('unique_organization_types', 0)}")
                        
                        top_types = td.get('top_organization_types', {})
                        if top_types:
                            print(f"\n       Top Types:")
                            for org_type, pct in list(top_types.items())[:5]:
                                print(f"         {org_type:15s} {pct:.2%}")
                
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Mapping (NEWEST) - Only for Strategy 2
    if 'strategy2' in dir_name:
        maps_dir = strategy_dir / 'maps'
        if maps_dir.exists():
            mapping_files = sorted(
                list(maps_dir.glob('*_mapping.json')),
                key=lambda x: x.stat().st_mtime, reverse=True
            )
            if mapping_files:
                print(f"\n📄 MAPPING: {mapping_files[0].name}")
                try:
                    with open(mapping_files[0], 'r') as f:
                        mapping_data = json.load(f)
                        if 'mappings' in mapping_data:
                            field = FIELD_CONFIG['strategy2_field']
                            if field in mapping_data['mappings']:
                                mappings = mapping_data['mappings'][field]
                                print(f"   Total mappings: {len(mappings)}")
                                print(f"\n   Sample Mappings (first 5):")
                                for item in mappings[:5]:
                                    orig = item.get('original', '')
                                    synth = item.get('synthetic', '')
                                    print(f"     {orig:<30} → {synth}")
                except Exception as e:
                    print(f"   ⚠️  Error: {e}")
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2
                print(f"   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 7: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports anonymized datasets for each strategy

**What you'll see (per strategy):**
- Available columns list
- Export path
- Preview (first 5 rows)
- Statistics (unique organizations, avg name length)

**Output structure:**
```
advanced_output/
├── strategy1_regional/
│   └── regional_organizations.csv
├── strategy2_industry/
│   └── industry_organizations.csv
└── strategy3_custom/
    └── custom_organizations.csv
```

**Note:** Files include both original and generated fields for comparison

In [ ]:
import os
from pathlib import Path

print("=" * 80)
print("📦 EXPORTING FINAL DATA FROM ALL STRATEGIES")
print("=" * 80)

# Get field config
try:
    field_s1 = FIELD_CONFIG["strategy1_field"]
    field_s2 = FIELD_CONFIG["strategy2_field"]
    field_s3 = FIELD_CONFIG["strategy3_field"]
except NameError:
    print("❌ Run Step 4 first!")
    raise

# Base export directory
export_base_dir = task_dir
task_dir.mkdir(parents=True, exist_ok=True)

print(f"\n📂 Export directory: {task_dir}\n")

# ============================================================================
# STRATEGY 1: Region-Based
# ============================================================================
if 'result_df_s1' in locals() and result_df_s1 is not None:
    print("=" * 80)
    print("📊 STRATEGY 1: REGION-BASED GENERATION")
    print("=" * 80)
    
    s1_dir = export_base_dir / 'strategy1_regional'
    s1_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s1 = f"{field_s1}_regional"
    
    if output_col_s1 in result_df_s1.columns:
        export_cols_s1 = ['record_id', field_s1, output_col_s1, 
                          'organization_type', 'region', 'industry']
        existing_cols_s1 = [col for col in export_cols_s1 if col in result_df_s1.columns]
        
        final_df_s1 = result_df_s1[existing_cols_s1].copy()
        
        output_path_s1 = s1_dir / 'regional_organizations.csv'
        try:
            final_df_s1.to_csv(output_path_s1, index=False)
            print(f"\n✅ Saved: {output_path_s1}")
            print(f"   Rows: {len(final_df_s1):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s1.head())
        
        print(f"\n📈 Statistics:")
        print(f"   Unique original:  {result_df_s1[field_s1].nunique()}")
        print(f"   Unique synthetic: {result_df_s1[output_col_s1].nunique()}")
        print(f"   Avg name length:  {result_df_s1[output_col_s1].str.len().mean():.1f} chars")
else:
    print("\n⚠️  Strategy 1 data not available")

# ============================================================================
# STRATEGY 2: Industry-Specific
# ============================================================================
if 'result_df_s2' in locals() and result_df_s2 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 2: INDUSTRY-SPECIFIC GENERATION")
    print("=" * 80)
    
    s2_dir = export_base_dir / 'strategy2_industry'
    s2_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s2 = f"{field_s2}_industry"
    
    if output_col_s2 in result_df_s2.columns:
        export_cols_s2 = ['record_id', field_s2, output_col_s2,
                          'organization_type', 'industry']
        existing_cols_s2 = [col for col in export_cols_s2 if col in result_df_s2.columns]
        
        final_df_s2 = result_df_s2[existing_cols_s2].copy()
        
        output_path_s2 = s2_dir / 'industry_organizations.csv'
        try:
            final_df_s2.to_csv(output_path_s2, index=False)
            print(f"\n✅ Saved: {output_path_s2}")
            print(f"   Rows: {len(final_df_s2):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s2.head())
        
        print(f"\n📈 Statistics:")
        print(f"   Unique original:  {result_df_s2[field_s2].nunique()}")
        print(f"   Unique synthetic: {result_df_s2[output_col_s2].nunique()}")
        print(f"   Avg name length:  {result_df_s2[output_col_s2].str.len().mean():.1f} chars")
else:
    print("\n⚠️  Strategy 2 data not available")

# ============================================================================
# STRATEGY 3: Custom Dictionaries
# ============================================================================
if 'result_df_s3' in locals() and result_df_s3 is not None:
    print("\n" + "=" * 80)
    print("📊 STRATEGY 3: CUSTOM DICTIONARIES")
    print("=" * 80)
    
    s3_dir = export_base_dir / 'strategy3_custom'
    s3_dir.mkdir(parents=True, exist_ok=True)
    
    output_col_s3 = f"{field_s3}_custom"
    
    if output_col_s3 in result_df_s3.columns:
        export_cols_s3 = ['record_id', field_s3, output_col_s3,
                          'organization_type', 'region']
        existing_cols_s3 = [col for col in export_cols_s3 if col in result_df_s3.columns]
        
        final_df_s3 = result_df_s3[existing_cols_s3].copy()
        
        output_path_s3 = s3_dir / 'custom_organizations.csv'
        try:
            final_df_s3.to_csv(output_path_s3, index=False)
            print(f"\n✅ Saved: {output_path_s3}")
            print(f"   Rows: {len(final_df_s3):,}")
        except PermissionError:
            print(f"\n⚠️  Cannot save (file may be open)")
        
        print(f"\n📊 Preview:")
        print(final_df_s3.head())
        
        print(f"\n📈 Statistics:")
        print(f"   Unique original:  {result_df_s3[field_s3].nunique()}")
        print(f"   Unique synthetic: {result_df_s3[output_col_s3].nunique()}")
        print(f"   Avg name length:  {result_df_s3[output_col_s3].str.len().mean():.1f} chars")
else:
    print("\n⚠️  Strategy 3 data not available")

# ============================================================================
# SUMMARY
# ============================================================================
print("\n" + "=" * 80)
print("✅ EXPORT COMPLETE")
print("=" * 80)
print(f"\n📂 Files saved to: {export_base_dir}")

if 'elapsed_s1' in locals():
    total_time = elapsed_s1 + elapsed_s2 + elapsed_s3
    print(f"⏱️  Total time: {total_time:.2f}s")

## 🎯 Summary

**Accomplished:**
- ✅ 3 strategies implemented and compared
- ✅ Multi-regional organization generation
- ✅ Industry-specific generation
- ✅ Custom dictionary-based generation
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Region-based strategy provides geographic diversity
- Industry-specific generation maintains domain context
- Custom dictionaries offer maximum control
- PRGN vs mapping: different consistency mechanisms for different needs
- All strategies preserve organization type characteristics

**Strategy recommendations:**
- **Use Strategy 1** when: Global datasets, regional diversity needed
- **Use Strategy 2** when: Industry context important, audit trails required
- **Use Strategy 3** when: Specialized domains, custom naming conventions

**Next steps:**
- Test with your own datasets
- Customize dictionaries for your domain
- Combine strategies for complex scenarios
- Deploy to production pipelines

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)